# OptiCrop — Model Training
Smart Agricultural Production Optimization Engine

This notebook loads the crop recommendation dataset, explores it, trains and
compares several classification algorithms (KNN, Logistic Regression,
Decision Tree, Random Forest), and saves the best-performing model and its
scaler for use by the Flask app (`app.py`).

**Dataset:** `../Dataset/Crop_recommendation.csv`

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style("whitegrid")
%matplotlib inline

## 2. Load Dataset

In [ ]:
df = pd.read_csv("../Dataset/Crop_recommendation.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning & Exploratory Analysis

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

In [ ]:
# Class balance across crop labels
plt.figure(figsize=(12, 5))
df['label'].value_counts().plot(kind='bar', color='#52753f')
plt.title("Number of Samples per Crop")
plt.xlabel("Crop")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation between numeric features
plt.figure(figsize=(8, 6))
sns.heatmap(df.drop(columns=['label']).corr(), annot=True, cmap='YlGn', fmt='.2f')
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
# Distribution of each feature
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(features):
    sns.histplot(df[col], kde=True, ax=axes[i], color='#3f5e3a')
    axes[i].set_title(col)
for j in range(len(features), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

## 4. Feature & Target Split

In [ ]:
FEATURE_ORDER = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

X = df[FEATURE_ORDER]
y = df['label']

X.shape, y.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

## 5. Feature Scaling
KNN and Logistic Regression are sensitive to feature scale, so we standardize the inputs. Tree-based models don't require it, but applying the same scaler to all keeps the pipeline consistent for deployment.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Train & Compare Models

In [ ]:
models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    cv_score = cross_val_score(model, X_train_scaled, y_train, cv=5).mean()
    results[name] = {"model": model, "test_accuracy": acc, "cv_accuracy": cv_score}
    print(f"{name:22s} | Test Accuracy: {acc:.4f} | 5-Fold CV Accuracy: {cv_score:.4f}")

In [ ]:
results_df = pd.DataFrame({
    name: {"Test Accuracy": r["test_accuracy"], "CV Accuracy": r["cv_accuracy"]}
    for name, r in results.items()
}).T

results_df.plot(kind='bar', figsize=(9, 5), color=['#52753f', '#c9a86a'])
plt.title("Model Comparison")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

results_df

## 7. Evaluate the Best Model

In [ ]:
best_name = max(results, key=lambda n: results[n]["test_accuracy"])
best_model = results[best_name]["model"]
print(f"Best model: {best_name} ({results[best_name]['test_accuracy']:.4f} test accuracy)")

In [ ]:
y_pred = best_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred, labels=best_model.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGn',
            xticklabels=best_model.classes_, yticklabels=best_model.classes_)
plt.title(f"Confusion Matrix — {best_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (only meaningful for tree-based models)
if hasattr(best_model, "feature_importances_"):
    importance = pd.Series(best_model.feature_importances_, index=FEATURE_ORDER).sort_values()
    importance.plot(kind='barh', figsize=(7, 4), color='#3f5e3a')
    plt.title(f"Feature Importance — {best_name}")
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_name} does not expose feature_importances_.")

## 8. Exploratory Clustering (K-Means)
Unsupervised look at how crops naturally group by soil & climate conditions — useful context for the research/policy scenario, not used in the deployed prediction pipeline.

In [ ]:
kmeans = KMeans(n_clusters=len(df['label'].unique()), random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_train_scaled)

plt.figure(figsize=(7, 5))
plt.scatter(X_train_scaled[:, 0], X_train_scaled[:, 3], c=clusters, cmap='viridis', alpha=0.6)
plt.xlabel("Nitrogen (scaled)")
plt.ylabel("Temperature (scaled)")
plt.title("K-Means Clusters (N vs Temperature)")
plt.tight_layout()
plt.show()

## 9. Save Model & Scaler for Deployment

In [ ]:
with open("../Flask/crop_recommendation_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

with open("../Flask/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Saved crop_recommendation_model.pkl and scaler.pkl to the Flask/ directory")

### Next step
The model and scaler are saved directly into `Flask/`, right next to `app.py` —
no manual copying needed. Just run the Flask app from the `Flask/` folder.